In [2]:
%pip install simfile pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 29.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 47.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import glob

# packs from my setup
songs_path = '/media/win/Users/Lenovo/AppData/Roaming/ITGmania/Songs/'
def my_setup(pack_name):
    return os.path.join(songs_path, pack_name)

def my_setup_srpg(which):
    return [
        my_setup(f'Stamina RPG {which}{suffix}')
        for suffix in ['', ' - SN', ' - FE']
    ]

def stash_path(pack_prefix):
    return glob.glob('./packs/' + pack_prefix + '*')

pack_paths = [
    *stash_path('Stamina RPG 6'),
    my_setup('Stamina RPG 6'),
    my_setup('SRPG6 NNP'),
    *stash_path('ECS11'),
    *my_setup_srpg(7),
    *stash_path('ECS12'),
    *my_setup_srpg(8),
    *stash_path('ECS13'),
    *my_setup_srpg(9),
    *stash_path('ECS14'),
    *my_setup_srpg(10),
]
# reverse chronological order (so newer ratings are prioritized)
pack_paths.reverse()

In [11]:
pack_paths

['./packs/Stamina RPG 6']

In [12]:
import simfile
from simfile.dir import SimfilePack
from simfile.types import Simfile
from analysis.analyzer import SongAnalyzer
from analysis.hash import get_hash
import pandas as pd

try:
    existing_df = pd.read_json('../public/dataset.json')
    seen_hashes = set(existing_df['hash'])
except FileNotFoundError:
    existing_df = None
    seen_hashes = set()

data = []

def add_song_data(pack: SimfilePack | None, song: Simfile):
    pack_name = pack.name if pack else ''
    title = song.title
    if song.subtitle:
        title += ' ' + song.subtitle

    song_analyzer = SongAnalyzer(song)
    for chart_analyzer in song_analyzer.chart_analyzers.values():
        chart = chart_analyzer.chart
        if (chart.stepstype or '').lower() != 'dance-single':
            continue
        diff = (chart.difficulty or '').lower()
        chart_title = title
        if diff != 'challenge':
            chart_title += f' ({diff.title()})'

        chart_hash = get_hash(song, chart_analyzer.chart)
        if chart_hash in seen_hashes:
            print(f'skipping {title} (dupe hash)')
            continue
        seen_hashes.add(chart_hash)
        print(f'analyzing {pack_name}/{title}')

        stream_info = chart_analyzer.get_stream_info()
        total_stream = stream_info['total_stream']
        total_break = stream_info['total_break']
        bpms = stream_info['bpms']
        avg_bpm = stream_info['avg_bpm']
        try:
            meter = int(chart_analyzer.chart.meter)
        except ValueError:
            print(song.title, ': invalid meter')
            continue

        if total_stream == 0:
            print(title, ': no stream')
        else:
            mult = stream_info['quant'] / 16
            data.append((
                title,
                pack_name,
                chart_hash,
                meter,
                round(avg_bpm * mult, 3),
                total_stream * mult,
                total_stream / (total_stream + total_break),
                round(bpms[0] * mult, 3),
                round(bpms[1] * mult, 3),
                stream_info['segments'],
                chart_analyzer.get_density_graph()
            ))

for pack_path in pack_paths:
    pack = SimfilePack(pack_path)
    for song in pack.simfiles(strict=False):
        add_song_data(pack, song)

for sim_path in glob.glob('./packs/_misc/*.sm'):
    song = simfile.open(sim_path, strict=False)
    add_song_data(None, song)

analyzing Stamina RPG 6/[20] [208] Fast Animu Music 2020 Reduxx FB 208
analyzing Stamina RPG 6/[20] [225] Abyssal
skipping [22] [244] lolivision (dupe hash)
analyzing Stamina RPG 6/[23] [225] Trails of Cold Stream the 3rd
skipping [24] [280] Never Ending Story (Hard) (Power Club Vocal Mix) (dupe hash)
analyzing Stamina RPG 6/[19] [204] 2 Crazy Brothers
analyzing Stamina RPG 6/[19] [200] Ha(Ne)rdcore OTAKU Collectionz Pt 1
analyzing Stamina RPG 6/[23] [225] Portal Of I Continuous Mix Archengrove Mk 2
analyzing Stamina RPG 6/[25] [280] Penguins Prank Party
analyzing Stamina RPG 6/[24] [260] Masochist
analyzing Stamina RPG 6/[21] [256] on stage!
skipping [18] [190] Infinity Engine (dupe hash)
analyzing Stamina RPG 6/[26] [296] Disconnected -Trance Mix-
analyzing Stamina RPG 6/[20] [230] Countdown
analyzing Stamina RPG 6/[24] [275] Chu Chu Lovely (Maximum The Hormone Cover)
analyzing Stamina RPG 6/[18] [210] Battle for Eternity
analyzing Stamina RPG 6/[19] [208] Solar Flare
analyzing Stami

In [13]:
columns = [
    'name', 'pack', 'hash', 'meter',
    'bpm', 'totalStream', 'streamDensity',
    'minBpm', 'maxBpm',
    'segments', 'densityGraph']
df = pd.DataFrame(data, columns=columns)
if existing_df is not None:
    df = pd.concat([existing_df, df], axis=0, ignore_index=True)
df

,name,pack,hash,meter,bpm,totalStream,streamDensity,minBpm,maxBpm,segments,densityGraph
0,[17] [210] Cavalcade,Stamina RPG 10 - FE,563fca1a7a893a623b8dec87bea86bf78f8b6d75,17,210.00,104.0,0.732394,210.00,210.00,"[5, -3, 14, -18, 15, -5, 1, -3, 32, -8, 7, 30]","[[0.066, 4.375], [1.2088571429, 3.5], [2.35171..."
1,[17] [217] in n out of LOVE,Stamina RPG 10 - FE,cd0db95728d86df32ce5118d35d8be257f06c015,17,217.45,95.0,0.536723,217.45,217.45,"[7, -55, 3, 15, 15, -7, 17, -15, 1, 15, 8, 14]","[[0.101, 8.1543748875], [1.2047020157000001, 7..."
2,[18] [215] 23,Stamina RPG 10 - FE,0db4b9c5a5fd04aa98df3d7312400f5eaf00cbc5,18,215.00,156.0,0.768473,215.00,215.00,"[1, 1, -4, 1, 33, -6, 34, -16, 17, 1, -4, 1, -...","[[1.108, 0.0], [8.9219534884, 0.0], [10.038232..."
3,[18] [210] Across the Frenchcore,Stamina RPG 10 - FE,383ae480eba710b569cdbbba5f8923c11ad53009,18,210.00,170.0,0.817308,210.00,210.00,"[7, 7, 7, 7, 15, 15, 15, 15, -29, 2, -2, 80]","[[-0.019, 0.875], [1.1238571429, 4.375], [2.26..."
4,[18] [226] Masshirona Kutsu,Stamina RPG 10 - FE,91a11bc5bcadad250afa826c05316de52ca53f36,18,226.00,67.0,0.770115,226.00,226.00,"[8, -4, 8, -2, 2, -4, 9, 20, -9, 20]","[[0.008, 0.0], [5.3177345133, 0.0], [6.3796814..."
...,...,...,...,...,...,...,...,...,...,...,...
1940,[19] [210] What Is A Tunecore,Stamina RPG 6,b5e4c087b6bbe08a9da867305444e1641d90177e,19,210.00,216.0,0.964286,210.00,210.00,"[136, -8, 80]","[[-1.142, 0.0], [0.001, 8.75], [1.144, 8.75], ..."
1941,[27] [290] Monstrum Spectrum,Stamina RPG 6,4c710adac4297c364e412ea1f8171eeff301bc35,27,290.00,98.0,0.970297,290.00,290.00,"[79, -3, 19]","[[-0.021, 14.5], [0.807, 19.333333333333332], ..."
1942,[22] [258] Kangokurakuou,Stamina RPG 6,fedc2d92d46975d68c87edf19d6339b26db499f0,22,258.00,70.0,0.673077,258.00,258.00,"[6, -2, 16, -15, 7, -16, 8, 33]","[[0.049, 0.0], [6.561, 0.0], [7.491, 17.200000..."
1943,[20] [222] Non-existence proof,Stamina RPG 6,75aebdba91c1686f46e062cc5b33c5c46cffc2cf,20,222.00,201.0,0.851695,222.00,222.00,"[3, -6, 14, 129, -16, 48, -2, 2, -4, 4, -6, 1]","[[0.600, 0.0], [2.762, 0.0], [3.843, 0.9250000..."


In [14]:
df.to_json('../public/dataset.json', orient='records', index=False)